<div style="background-color:#e6f2ff; padding:20px; border-radius:10px;">
<img style="float:left; margin-right:20px;" src='Figures/alinco.png' width="120"/>
<h1 style="color:#000047;">Actividad 5: Regresión Logística para reviews de Peliculas</h1>
<br style="clear:both"/>
</div>

<div style="border-left:4px solid #000047; padding:10px; margin-top:10px; background:#f5f5f5;">
<b>Objetivo:</b> En esta actividad se abordará la implementación de un modelo de aanálisis de sentimientos para peliculas. 
</div>

<div style="margin-top:10px;">
<b>Instrucciones generales:</b>
<ul>
<li>Considere el review de peliculas del archivo "moviereviews.tsv" que se encuentr en la carpeta Data</li>
<li>Cree un modelo de Regresión Logística para clasificación de sentimientos con las funciones vistas en clase.</li>

</ul>
</div>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

# Cargar dataset 
df = pd.read_csv('Data/moviereviews.tsv', sep='\t', quoting=1)
df = df[['label', 'review']].dropna()
df['review'] = df['review'].astype(str)
df['y'] = (df['label'].str.lower() == 'pos').astype(int)

print('Tamano dataset:', len(df))
print('Distribucion de clases:')
print(df['label'].value_counts())
df.head(3)

Tamano dataset: 1965
Distribucion de clases:
label
neg    983
pos    982
Name: count, dtype: int64


,label,review,y
0,neg,how do films like mouse hunt get into theatres...,0
1,neg,some talented actresses are blessed with a dem...,0
2,pos,this has been an extraordinary year for austra...,1


In [2]:
class RegresionLogisticaSentimiento:
    """Regresion logistica binaria con gradiente descendente batch."""

    def __init__(self, lr=0.3, epochs=500, lambda_=0.01, umbral=0.5):
        self.lr = lr
        self.epochs = epochs
        self.lambda_ = lambda_
        self.umbral = umbral
        self.w = None
        self.b = 0.0
        self.historial = []

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def _perdida(self, y_hat, y):
        eps = 1e-15
        y_hat = np.clip(y_hat, eps, 1 - eps)
        ce = -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
        return ce + (self.lambda_ / 2) * np.dot(self.w, self.w)

    def fit(self, X, y, verbose=True):
        N, D = X.shape
        self.w = np.zeros(D)
        self.b = 0.0
        self.historial = []

        for epoch in range(self.epochs):
            y_hat = self._sigmoid(X.dot(self.w) + self.b)
            loss = self._perdida(y_hat, y)
            self.historial.append(loss)

            error = y_hat - y
            grad_w = X.T.dot(error) / N + self.lambda_ * self.w
            grad_b = np.mean(error)

            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

            if verbose and (epoch % 100 == 0 or epoch == self.epochs - 1):
                acc = accuracy_score(y, self.predecir(X))
                print(f'Epoch {epoch:4d} | Perdida: {loss:.4f} | Acc train: {acc:.4f}')

        return self

    def predecir_proba(self, X):
        return self._sigmoid(X.dot(self.w) + self.b)

    def predecir(self, X):
        return (self.predecir_proba(X) >= self.umbral).astype(int)

    def palabras_importantes(self, vocab, top_n=10):
        idx_pos = np.argsort(self.w)[::-1][:top_n]
        idx_neg = np.argsort(self.w)[:top_n]
        top_pos = [(vocab[i], float(self.w[i])) for i in idx_pos]
        top_neg = [(vocab[i], float(self.w[i])) for i in idx_neg]
        return top_pos, top_neg

In [ ]:
# Vectorizacion TF-IDF 
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X = vectorizer.fit_transform(df['review']).toarray()
y = df['y'].values.astype(int)
vocabulario = vectorizer.get_feature_names_out()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)

modelo = RegresionLogisticaSentimiento(lr=0.3, epochs=500, lambda_=0.01)
modelo.fit(X_train, y_train, verbose=True)

y_pred = modelo.predecir(X_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('\n=== Metricas en test ===')
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1-score:  {f1:.4f}')
print('\nReporte de clasificacion:')
print(classification_report(y_test, y_pred, target_names=['negativo', 'positivo']))

Train: (1473, 134903) Test: (492, 134903)
Epoch    0 | Perdida: 0.6931 | Acc train: 0.8900
Epoch  100 | Perdida: 0.6865 | Acc train: 0.9728
Epoch  200 | Perdida: 0.6829 | Acc train: 0.9722
Epoch  300 | Perdida: 0.6810 | Acc train: 0.9715
Epoch  400 | Perdida: 0.6800 | Acc train: 0.9722
Epoch  499 | Perdida: 0.6794 | Acc train: 0.9722

=== Metricas en test ===
Accuracy:  0.8476
Precision: 0.8670
Recall:    0.8211
F1-score:  0.8434

Reporte de clasificacion:
              precision    recall  f1-score   support

    negativo       0.83      0.87      0.85       246
    positivo       0.87      0.82      0.84       246

    accuracy                           0.85       492
   macro avg       0.85      0.85      0.85       492
weighted avg       0.85      0.85      0.85       492



In [ ]:
# Palabras mas asociadas a cada clase segun los pesos aprendidos
top_pos, top_neg = modelo.palabras_importantes(vocabulario, top_n=15)

print('Top palabras positivas:')
for w, peso in top_pos:
    print(f'  {w:20s} {peso: .4f}')

print('\nTop palabras negativas:')
for w, peso in top_neg:
    print(f'  {w:20s} {peso: .4f}')

# Prueba
reviews_nuevas = [
    'I loved this movie, the acting was fantastic and the story was moving',
    'Terrible film. Boring plot and bad performances, I regret watching it'
]

X_new = vectorizer.transform(reviews_nuevas).toarray()
probas = modelo.predecir_proba(X_new)
preds = modelo.predecir(X_new)

for txt, p, yhat in zip(reviews_nuevas, probas, preds):
    etiqueta = 'positivo' if yhat == 1 else 'negativo'
    print('\nReview:', txt)
    print(f'Prob(positivo): {p:.4f} -> Prediccion: {etiqueta}')

Top palabras positivas:
  life                  0.0715
  world                 0.0561
  he is                 0.0555
  as the                0.0514
  also                  0.0495
  american              0.0494
  family                0.0487
  perfect               0.0475
  different             0.0461
  the best              0.0453
  performance           0.0452
  oscar                 0.0441
  many                  0.0439
  great                 0.0437
  true                  0.0436

Top palabras negativas:
  bad                  -0.1225
  worst                -0.0816
  stupid               -0.0628
  boring               -0.0618
  the worst            -0.0612
  nothing              -0.0590
  plot                 -0.0588
  waste                -0.0566
  have                 -0.0552
  why                  -0.0548
  script               -0.0531
  looks                -0.0507
  should have          -0.0506
  movie                -0.0497
  supposed             -0.0474

Review: I loved this